<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://campusvirtual.urv.cat/pluginfile.php/1/core_admin/logocompact/300x300/1767733831/logoURVppd.png", align="left">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">2025-2026 SCIENTIFIC PROGRAMMING (17715101)</p>
<p style="margin: 0; text-align:right;">MD Health Data Science / Biomedical Data Science</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Preprocessing

In [1]:
### Load relevant packages

import pandas                  as pd
import numpy                   as np
import matplotlib.pyplot       as plt
import seaborn                 as sns
from sklearn.preprocessing import MinMaxScaler


# This statement allow to display plots without asking to 
%matplotlib inline

# Set style for plots
plt.style.use('ggplot')

In [2]:
# Adding src to the current path
import sys
sys.path.append('..')

Normalization

From exploratory data analysis step, the z-score standardization was chosen to bring all variables of different scales to a mean of 0 and standard deviation of 1.

In [3]:
#Load scaled dataset
df = pd.read_csv('../data/breast_cancer_renamed.csv', dtype = { 'diagnosis': 'category', 'diagnosis_label': 'category'}) # indicate that diagnosis is categorical
df.head()

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis,diagnosis_label
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant


Feature Selection

From previous analysis, it can be observed that there are many pairs of variables that are highly correlated. This redundancy is unnecessary for our prediction task. Hence, for any pairs of variables that have correlation coefficient of above 0.9, only one variable is kept. Only 20 features are kept for predictive modeling task.

In [4]:
# Load highly correlated variables
high_corr_var = pd.read_csv('../results/highly_correlated_variables.csv')

# Keep variables in feature_1 and drop all variables in feature_2
var_to_drop_list = high_corr_var['feature_2'].unique().tolist()

# Drop highly correlated variables (10 variables)
df_reduced = df.drop(columns = var_to_drop_list)
df_reduced.to_csv('../data/breast_cancer_reduced.csv', index = False)

Aggregate

In [9]:
df_reduced = pd.read_csv('../data/breast_cancer_reduced.csv')
df_reduced.head()

,mean_radius,mean_texture,mean_smoothness,mean_compactness,mean_concavity,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,smoothness_error,...,concave_points_error,symmetry_error,fractal_dimension_error,worst_smoothness,worst_compactness,worst_concavity,worst_symmetry,worst_fractal_dimension,diagnosis,diagnosis_label
0,17.99,10.38,0.11840,0.27760,0.3001,0.2419,0.07871,1.0950,0.9053,0.006399,...,0.01587,0.03003,0.006193,0.1622,0.6656,0.7119,0.4601,0.11890,0,malignant
1,20.57,17.77,0.08474,0.07864,0.0869,0.1812,0.05667,0.5435,0.7339,0.005225,...,0.01340,0.01389,0.003532,0.1238,0.1866,0.2416,0.2750,0.08902,0,malignant
2,19.69,21.25,0.10960,0.15990,0.1974,0.2069,0.05999,0.7456,0.7869,0.006150,...,0.02058,0.02250,0.004571,0.1444,0.4245,0.4504,0.3613,0.08758,0,malignant
3,11.42,20.38,0.14250,0.28390,0.2414,0.2597,0.09744,0.4956,1.1560,0.009110,...,0.01867,0.05963,0.009208,0.2098,0.8663,0.6869,0.6638,0.17300,0,malignant
4,20.29,14.34,0.10030,0.13280,0.1980,0.1809,0.05883,0.7572,0.7813,0.011490,...,0.01885,0.01756,0.005115,0.1374,0.2050,0.4000,0.2364,0.07678,0,malignant


In [14]:
df_reduced.columns

Index(['mean_radius', 'mean_texture', 'mean_smoothness', 'mean_compactness',
       'mean_concavity', 'mean_symmetry', 'mean_fractal_dimension',
       'radius_error', 'texture_error', 'smoothness_error',
       'compactness_error', 'concavity_error', 'concave_points_error',
       'symmetry_error', 'fractal_dimension_error', 'worst_smoothness',
       'worst_compactness', 'worst_concavity', 'worst_symmetry',
       'worst_fractal_dimension', 'diagnosis', 'diagnosis_label'],
      dtype='object')

In [ ]:
cols = df_reduced.columns.difference(["diagnosis", "diagnosis_label"])
aggregated_df = df_reduced[cols].groupby('mean_radius').mean().reset_index()
aggregated_df.head()

,mean_radius,compactness_error,concave_points_error,concavity_error,fractal_dimension_error,mean_compactness,mean_concavity,mean_fractal_dimension,mean_smoothness,mean_symmetry,mean_texture,radius_error,smoothness_error,symmetry_error,texture_error,worst_compactness,worst_concavity,worst_fractal_dimension,worst_smoothness,worst_symmetry
0,6.981,0.010840,0.000000,0.00000,0.004100,0.07568,0.00000,0.07818,0.11700,0.1930,13.43,0.2241,0.010190,0.02659,1.5080,0.12020,0.0000,0.09382,0.15840,0.2932
1,7.691,0.064570,0.013640,0.09252,0.007551,0.11990,0.09252,0.07751,0.08668,0.2037,25.44,0.2196,0.015470,0.02105,1.4790,0.30640,0.3393,0.10660,0.15960,0.2790
2,7.729,0.009692,0.000000,0.00000,0.006872,0.04878,0.00000,0.07285,0.08098,0.1870,25.49,0.3777,0.012660,0.02882,1.4620,0.08340,0.0000,0.09938,0.12560,0.3058
3,7.760,0.004660,0.000000,0.00000,0.002783,0.04362,0.00000,0.05884,0.05263,0.1587,24.54,0.3857,0.007189,0.02676,1.4280,0.06444,0.0000,0.07039,0.08996,0.2871
4,8.196,0.016460,0.005917,0.01588,0.002582,0.05943,0.01588,0.06503,0.08600,0.1769,16.84,0.1563,0.008968,0.02574,0.9567,0.13570,0.0688,0.07409,0.12970,0.3105
